In [2]:
import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

base_path = "/data2/2104/derivatives/deepmreye/"

gaze_file = os.path.join(base_path, "evals/eval_median_5_v2_sub-TR.txt")
gaze_data = {}

with open(gaze_file, 'r') as f:
    lines = f.readlines()

curr_run = None
for line in lines:
    if line.startswith("RUN:"):
        full_path = line.strip().split("RUN: ")[1]
        run_id = os.path.basename(full_path).replace('.npz', '')
        curr_run = run_id
        gaze_data[curr_run] = []
    elif line.strip() == "" or line.startswith("TR"):
        continue
    else:
        parts = line.strip().split()
        if len(parts) >= 3:
            try:
                x, y = float(parts[1]), float(parts[2])
                gaze_data[curr_run].append([x, y])
            except ValueError:
                continue

# convert lists to numpy arrays
for run in gaze_data:
    gaze_data[run] = np.array(gaze_data[run])

# Load movie times
movie_times_path = os.path.join(base_path, "movie_times.csv")
movie_times = pd.read_csv(movie_times_path)

def compute_isc_for_movie(movie_name):
    segments = []
    
    for idx, row in movie_times.iterrows():
        if row['movie'] != movie_name:
            continue
        
        run_id = f"sub-{row['sub']}_ses-{row['ses']}_run-{row['run']}"
        
        if run_id not in gaze_data:
            continue
        
        tr_start = row['TR_start'] - 1
        tr_end = row['TR_end']
        
        # Check if TR range is valid
        if tr_start < 0:
            tr_start = 0
            
        if tr_end > len(gaze_data[run_id]):
            tr_end = len(gaze_data[run_id])
            
        if tr_start >= tr_end:
            continue
        
        seg = gaze_data[run_id][tr_start:tr_end, :]
        segments.append(seg)
    
    if len(segments) < 2:
        return None
    
    # Make sure all segments have same length (shortest one)
    min_len = min(seg.shape[0] for seg in segments)
    segments = [seg[:min_len] for seg in segments]
    
    # Stack into array: (n_subjects, n_TRs, 2)
    segments = np.stack(segments)
    
    # Compute ISC using leave-one-out for each coordinate
    n_subs, n_TRs, n_dim = segments.shape
    isc = np.zeros(n_dim)
    
    for dim in range(n_dim):
        vals = segments[:, :, dim]
        corrs = []
        for i in range(n_subs):
            others_mean = np.mean(np.delete(vals, i, axis=0), axis=0)
            r, _ = pearsonr(vals[i], others_mean)
            corrs.append(r)
        isc[dim] = np.mean(corrs)
    
    return isc

# Get unique movies
movies = movie_times['movie'].unique()
isc_results = {}

for movie in movies:
    result = compute_isc_for_movie(movie)
    if result is not None:
        isc_results[movie] = result

# Print results
for movie, isc_val in isc_results.items():
    print(f"{movie}: X={isc_val[0]:.3f}, Y={isc_val[1]:.3f}")

PlaceOfMyBirth: X=0.379, Y=0.326
FutureBoyfriend: X=0.448, Y=0.348
Scrambled: X=0.458, Y=0.438
Oxygen: X=0.416, Y=0.382
AnObjectAtRest: X=0.333, Y=0.453
InnerWorkings: X=0.269, Y=0.337
BorrowedTime: X=0.131, Y=0.184
StrayDogs: X=0.501, Y=0.355
Paperman: X=0.384, Y=0.147


In [1]:
import os, re
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional
from scipy.stats import pearsonr

# --- paths (adjust if needed) ---
EVAL_TXT = "/data2/2104/derivatives/deepmreye/evals/eval_median_5_v2_sub-TR.txt"
MOVIE_TIMES_CSV = "/data2/2104/derivatives/deepmreye/movie_times.csv"

# Enable per-segment z-scoring (recommended for between-subject comparability of X/Y coordinates)
ZSCORE_SEGMENTS = True

# --- drug mapping (use your existing function if present) ---
try:
    drug_guesses  # type: ignore
except NameError:
    def drug_guesses() -> Dict[str, str]:
        return {
            '21040005':'4','21040016':'4','21040017':'3','21040021':'4',
            '21040035':'3','21040033':'3','21040036':'3','21040040':'3',
            '21040042':'4','21040047':'4','21040045':'4','21040053':'4',
            '21040052':'3','21040057':'3','21040060':'3','21040066':'4'
        }

def map_cond(sub: str, ses: str) -> Optional[str]:
    dg = drug_guesses()
    ses_psil = dg.get(sub)
    if ses_psil is None:
        print(f"⚠️ Subject {sub} not in drug_guesses() list — skipping.")
        return None
    return 'psil' if ses_psil == ses else 'pla'

# --- parse DeepMReye eval file into run_id -> (T,2) [X,Y] ---
_RUN_RE = re.compile(r"sub-(\d+)_ses-(\d+)_run-(\d+)\.npz", re.IGNORECASE)
def parse_eval_file(eval_txt_path: str) -> Dict[str, np.ndarray]:
    data: Dict[str, List[List[float]]] = {}
    current = None
    with open(eval_txt_path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("RUN:"):
                m = _RUN_RE.search(line)
                if m:
                    sub, ses, run = m.group(1), m.group(2), m.group(3)
                    current = f"sub-{sub}_ses-{ses}_run-{run}"
                    data[current] = []
                else:
                    current = None
            elif line.startswith("TR"):
                continue
            else:
                if current is None:
                    continue
                parts = re.split(r"[\t ]+", line)
                if len(parts) >= 3:
                    try:
                        x, y = float(parts[1]), float(parts[2])
                        data[current].append([x, y])
                    except ValueError:
                        pass
    return {k: (np.array(v, float) if v else np.empty((0,2))) for k, v in data.items()}

# --- safe CSV reader & column normalizer ---
def load_movie_times(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, sep=None, engine="python")
    df = df.loc[:, ~df.columns.str.match(r"^\s*Unnamed|^\s*$", na=False)]
    df.columns = [c.strip() for c in df.columns]
    df.columns = [re.sub(r"[ \t\xa0]+", "", c) for c in df.columns]
    return df

def column_getter(df: pd.DataFrame):
    norm_map = {c.lower(): c for c in df.columns}
    def col(name: str) -> str:
        key = name.lower()
        if key in norm_map:
            return norm_map[key]
        key2 = key.replace("_", "")
        for k in norm_map:
            if k.replace("_","") == key2:
                return norm_map[k]
        raise KeyError(name)
    return col

# --- ISC helpers ---
def zscore_1d(x: np.ndarray) -> np.ndarray:
    mu, sd = np.nanmean(x), np.nanstd(x)
    return (x - mu)/sd if (sd not in (0, np.nan)) else (x - mu)

def fisher_z(r: np.ndarray) -> np.ndarray:
    r = np.clip(r, -0.999999, 0.999999)
    return 0.5*np.log((1+r)/(1-r))

def fisher_r(z: float) -> float:
    return (np.exp(2*z)-1)/(np.exp(2*z)+1)

def pairwise_mean_isc(mat: np.ndarray) -> Tuple[float, int]:
    if mat.shape[0] < 2:
        return np.nan, 0
    C = np.corrcoef(mat)
    iu = np.triu_indices_from(C, 1)
    z = fisher_z(C[iu])
    return fisher_r(np.nanmean(z)), len(z)

def loo_isc(mat: np.ndarray) -> float:
    n, T = mat.shape
    if n < 2:
        return np.nan
    vals = []
    for i in range(n):
        ref = np.nanmean(np.delete(mat, i, axis=0), axis=0)
        r, _ = pearsonr(mat[i], ref)
        vals.append(r)
    return float(np.nanmean(vals))

In [ ]:
# --- BETWEEN-GROUP ISC helpers ---

def mean_ts(mat: np.ndarray) -> np.ndarray:
    """Mean over subjects (axis=0). mat: N x T"""
    return np.nanmean(mat, axis=0)

def corr_1d(a: np.ndarray, b: np.ndarray) -> float:
    r, _ = pearsonr(a, b)
    return float(r)

def loo_between_groups(A: np.ndarray, B: np.ndarray) -> float:
    """
    Method 1 (LOO Between Groups):
    For each subject in A, corr with mean(B); for each subject in B, corr with mean(A).
    Returns Fisher-r-to-z averaged value in r-space.
    """
    if A.shape[0] < 1 or B.shape[0] < 1:
        return np.nan
    mA, mB = mean_ts(A), mean_ts(B)
    vals = [corr_1d(a, mB) for a in A] + [corr_1d(b, mA) for b in B]
    return fisher_r(np.nanmean(fisher_z(np.array(vals))))

def pairwise_between_groups(A: np.ndarray, B: np.ndarray) -> Tuple[float, int]:
    """
    Method 2 (All Cross-Group Pairs):
    Returns (grand-average r, number of cross pairs).
    """
    nA, nB = A.shape[0], B.shape[0]
    if nA < 1 or nB < 1:
        return np.nan, 0
    zvals = []
    for i in range(nA):
        for j in range(nB):
            zvals.append(fisher_z(corr_1d(A[i], B[j])))
    return fisher_r(np.nanmean(zvals)), nA * nB


In [ ]:
# --- load everything ---
runs_xy = parse_eval_file(EVAL_TXT)
mt = load_movie_times(MOVIE_TIMES_CSV)
col = column_getter(mt)

# verify required columns exist (case/underscore-insensitive)
required = ["sub","ses","run","movie","TR_start","TR_end"]
missing = [c for c in required if c not in {k.lower() for k in mt.columns} 
           and all(c.lower().replace("_","") != k.lower().replace("_","") for k in mt.columns)]
if missing:
    raise ValueError(f"movie_times.csv missing columns (case/underscore-insensitive): {missing}")

# attach condition
mt = mt.copy()
mt["cond"] = [map_cond(str(r[col("sub")]), str(r[col("ses")])) for _, r in mt.iterrows()]
mt = mt[mt["cond"].isin(["pla", "psil"])].copy()

# --- build segments per condition/movie ---
by_cond_movie = {"pla": {}, "psil": {}}
for _, r in mt.iterrows():
    sub = str(r[col("sub")]); ses = str(r[col("ses")]); run = str(r[col("run")]); mov = str(r[col("movie")])
    tr_s = int(np.floor(float(r[col("TR_start")]) - 1))  # 0-based inclusive
    tr_e = int(np.floor(float(r[col("TR_end")])))        # exclusive
    run_id = f"sub-{sub}_ses-{ses}_run-{run}"
    arr = runs_xy.get(run_id)
    if arr is None or arr.size == 0:
        continue
    tr_s = max(0, tr_s); tr_e = min(tr_e, arr.shape[0])
    if tr_s >= tr_e:
        continue
    seg = arr[tr_s:tr_e, :]  # (T,2)
    if ZSCORE_SEGMENTS:
        seg = np.column_stack([zscore_1d(seg[:,0]), zscore_1d(seg[:,1])])
    cond = map_cond(sub, ses)
    by_cond_movie.setdefault(cond, {}).setdefault(mov, []).append(seg)

# --- compute per-movie, within-cond ISC (PAIRWISE + LOO) ---
rows = []
for cond, movies in by_cond_movie.items():
    for mov, segs in movies.items():
        if len(segs) < 2:
            continue
        lengths = [s.shape[0] for s in segs]
        #print(f"{mov} ({cond}) segment lengths per subject: {lengths}")

        min_len = min(lengths)
        segs = [s[:min_len] for s in segs]
        X = np.vstack([s[:,0] for s in segs])  # N x T
        Y = np.vstack([s[:,1] for s in segs])

        n_subs = X.shape[0]
        n_pairs = n_subs * (n_subs - 1) // 2

        pwX, _ = pairwise_mean_isc(X); pwY, _ = pairwise_mean_isc(Y)
        pwM = np.nanmean([pwX, pwY])

        looX = loo_isc(X); looY = loo_isc(Y)
        looM = np.nanmean([looX, looY])

        rows.append({
            "cond": cond, "movie": mov, "n_subjects": n_subs, "n_pairs": n_pairs,
            "T_used": min_len,
            "pair_X": pwX, "pair_Y": pwY, "pair_meanXY": pwM,
            "loo_X": looX, "loo_Y": looY, "loo_meanXY": looM
        })

per_movie = pd.DataFrame(rows).sort_values(["cond","movie"]).reset_index(drop=True)

In [18]:
# --- compute per-movie, BETWEEN-group ISC (LOO + Pairwise) ---

rows_between = []
all_movies = set(list(by_cond_movie.get("pla", {}).keys()) + list(by_cond_movie.get("psil", {}).keys()))
for mov in sorted(all_movies):
    segs_pla = by_cond_movie.get("pla", {}).get(mov, [])
    segs_psi = by_cond_movie.get("psil", {}).get(mov, [])
    if len(segs_pla) == 0 or len(segs_psi) == 0:
        continue

    # Equalize segment length across *both* groups for this movie
    lengths = [s.shape[0] for s in (segs_pla + segs_psi)]
    min_len = min(lengths)
    segs_pla = [s[:min_len] for s in segs_pla]
    segs_psi = [s[:min_len] for s in segs_psi]

    # Build channel matrices
    X_pla = np.vstack([s[:,0] for s in segs_pla])  # Np x T
    Y_pla = np.vstack([s[:,1] for s in segs_pla])
    X_psi = np.vstack([s[:,0] for s in segs_psi])  # Nq x T
    Y_psi = np.vstack([s[:,1] for s in segs_psi])

    n_pla, n_psi = X_pla.shape[0], X_psi.shape[0]
    n_pairs_cross = n_pla * n_psi

    # Method 1: LOO between groups (channel-wise + meanXY)
    looX = loo_between_groups(X_pla, X_psi)
    looY = loo_between_groups(Y_pla, Y_psi)
    looM = np.nanmean([looX, looY])

    # Method 2: Pairwise between groups (channel-wise + meanXY)
    pwX, _ = pairwise_between_groups(X_pla, X_psi)
    pwY, _ = pairwise_between_groups(Y_pla, Y_psi)
    pwM = np.nanmean([pwX, pwY])

    rows_between.append({
        "movie": mov,
        "n_pla": n_pla, "n_psil": n_psi, "n_pairs_cross": n_pairs_cross,
        "T_used": min_len,
        "btwn_pair_X": pwX, "btwn_pair_Y": pwY, "btwn_pair_meanXY": pwM,
        "btwn_LOO_X": looX, "btwn_LOO_Y": looY, "btwn_LOO_meanXY": looM,
    })

per_movie_between = pd.DataFrame(rows_between).sort_values("movie").reset_index(drop=True)

In [16]:
def summarize(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    # TR-weighted mean across movies per condition
    weights = df.groupby("cond")["T_used"].sum()
    def wmean(col):
        return (df["T_used"] * df[col]).groupby(df["cond"]).sum() / weights
    weighted = pd.DataFrame({
        "pair_X": wmean("pair_X"),
        "pair_Y": wmean("pair_Y"),
        "pair_meanXY": wmean("pair_meanXY"),
        "loo_X":  wmean("loo_X"),
        "loo_Y":  wmean("loo_Y"),
        "loo_meanXY":  wmean("loo_meanXY"),
    })
    return weighted

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print("=== Per-movie ISC (segmented by movie_times), within each condition ===")
print(per_movie.to_string(index=False))

print("\n=== Across-movie summaries (TR-weighted, per condition) ===")
print(summarize(per_movie).to_string())

=== Per-movie ISC (segmented by movie_times), within each condition ===
cond           movie  n_subjects  n_pairs  T_used  pair_X  pair_Y  pair_meanXY  loo_X  loo_Y  loo_meanXY
 pla  AnObjectAtRest          16      120     366  0.0778  0.1056       0.0917 0.2110 0.2623      0.2367
 pla    BorrowedTime          16      120     441  0.0641  0.0811       0.0726 0.1820 0.2170      0.1995
 pla FutureBoyfriend          16      120     756  0.2486  0.1350       0.1918 0.4537 0.3073      0.3805
 pla   InnerWorkings          16      120     444  0.0618  0.1457       0.1038 0.1758 0.3244      0.2501
 pla          Oxygen          16      120     755  0.1601  0.1737       0.1669 0.3446 0.3629      0.3537
 pla        Paperman          16      120     455  0.1805  0.1469       0.1637 0.3720 0.3263      0.3491
 pla  PlaceOfMyBirth          16      120     339  0.1848  0.1348       0.1598 0.3774 0.3075      0.3425
 pla       Scrambled          16      120     340  0.1528  0.1771       0.1649 0.3336 0.

In [19]:
def summarize_between(df: pd.DataFrame) -> pd.Series:
    if df.empty:
        return pd.Series(dtype=float)
    w = df["T_used"]
    out = {}
    for col in ["btwn_pair_X","btwn_pair_Y","btwn_pair_meanXY",
                "btwn_LOO_X","btwn_LOO_Y","btwn_LOO_meanXY"]:
        out[col] = (w * df[col]).sum() / w.sum()
    return pd.Series(out)

print("\n=== Per-movie BETWEEN-group ISC (Placebo vs Psilocybin) ===")
if per_movie_between.empty:
    print("No movies with both groups present.")
else:
    print(per_movie_between.to_string(index=False))

print("\n=== Across-movie summary (TR-weighted) — BETWEEN-group ===")
print(summarize_between(per_movie_between).to_string())


=== Per-movie BETWEEN-group ISC (Placebo vs Psilocybin) ===
          movie  n_pla  n_psil  n_pairs_cross  T_used  btwn_pair_X  btwn_pair_Y  btwn_pair_meanXY  btwn_LOO_X  btwn_LOO_Y  btwn_LOO_meanXY
 AnObjectAtRest     16      15            240     366       0.1070       0.1554            0.1312      0.2710      0.3444           0.3077
   BorrowedTime     16      16            256     439       0.0764       0.0964            0.0864      0.2073      0.2486           0.2280
FutureBoyfriend     16      16            256     756       0.1960       0.1180            0.1570      0.4033      0.2890           0.3462
  InnerWorkings     16      15            240     444       0.0763       0.1493            0.1128      0.2142      0.3365           0.2753
         Oxygen     16      15            240     755       0.1671       0.1792            0.1732      0.3605      0.3787           0.3696
       Paperman     16      16            256     455       0.1907       0.1844            0.1876      0.